# Atelier 2.A — Pipeline RAG complet ★

**Durée : 75 min** · Projet fil rouge, incrément du Jour 2

Au Jour 1, le modèle inventait les délais de livraison de Fictiva. Vous construisez maintenant un **assistant documentaire** qui répond à partir de la documentation de l'entreprise, en citant ses sources :

lecture des documents → découpage → embeddings → index → recherche → génération citée.

Toutes les étapes tournent en local, sauf la génération : embeddings avec FastEmbed (modèle open source), index Qdrant en mémoire. Le pipeline obtenu est rangé dans `rag.py`, réutilisé par les ateliers 2.B à 2.D.

In [ ]:
from pathlib import Path
import json
import pandas as pd

import outils_llm as llm
import rag

MODELE = llm.choisir_modele()
print("Modèle de génération :", MODELE)

## 1. Le corpus

Dix documents de Fictiva SA, dans des formats variés. Les métadonnées indiquent le titre et la **confidentialité** de chaque document : la procédure interne du service après-vente ne doit jamais être montrée à un client.

In [ ]:
metadonnees = json.loads((rag.DOSSIER_CORPUS / "metadonnees.json").read_text(encoding="utf-8"))
pd.DataFrame([
    {"fichier": fichier, "format": Path(fichier).suffix, **meta,
     "taille_ko": round((rag.DOSSIER_CORPUS / fichier).stat().st_size / 1024, 1)}
    for fichier, meta in metadonnees.items()
])

## 2. Sans RAG, le modèle invente

Rappel du Jour 1 : sans accès à la documentation, le modèle répond avec assurance… et invente.

In [ ]:
for question in ["Livrez-vous en France ?", "Quel est le montant minimum d'une commande chez Fictiva ?"]:
    r = llm.demander(question + " Réponds en deux phrases.", systeme="Tu es l'assistant du service client de Fictiva SA.", modele=MODELE)
    print(f"Q : {question}\nR : {r.texte}\n")

## 3. Lire les documents

Chaque format pose ses problèmes. Pour les documents Word et HTML, MarkItDown produit un Markdown propre, avec titres et tableaux.

In [ ]:
from markitdown import MarkItDown

catalogue = MarkItDown().convert(str(rag.DOSSIER_CORPUS / "catalogue_produits.docx")).markdown
print(catalogue[:900])

Le PDF est plus délicat : il ne contient que des caractères positionnés sur une page, sans notion de titre ni de tableau. Comparons trois façons de lire le tableau des niveaux de maintenance.

In [ ]:
from pdfminer.high_level import extract_text

pdf = rag.DOSSIER_CORPUS / "conditions_maintenance.pdf"
lectures = {
    "MarkItDown": MarkItDown().convert(str(pdf)).markdown,
    "texte brut (pdfminer)": extract_text(str(pdf)),
    "mise en page conservée (pdfplumber)": rag.lire_pdf(pdf),
}
for nom, texte in lectures.items():
    debut = texte.find("Niveau")
    print(f"===== {nom}\n{texte[debut:debut + 520]}\n")

**À observer.** MarkItDown invente des tableaux à partir de la mise en page ; le texte brut sépare les colonnes, si bien que « Argent » n'est plus sur la même ligne que « 8 heures ouvrées » ; la lecture avec mise en page garde les lignes du tableau alignées. C'est elle que `rag.lire_pdf` utilise.

Pour des documents complexes (PDF scannés, tableaux imbriqués), on utilise des outils spécialisés comme Docling ou Unstructured. Dans tous les cas : **vérifiez le texte extrait de vos documents avant de les indexer**.

In [ ]:
documents = rag.charger_corpus()
for d in documents:
    print(f"{d.source:30} {d.confidentialite:8} {len(d.texte):6} caractères")

## 4. Découper

Un document entier est trop long pour être comparé finement à une question : on le découpe en **extraits** (*chunks*). Le découpage récursif coupe d'abord entre les paragraphes, puis entre les lignes, puis entre les phrases, pour rester sous la taille visée. Le chevauchement évite de couper une information en deux.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

decoupeur = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100, separators=["\n\n", "\n", ". ", " ", ""])
morceaux = decoupeur.split_text(documents[0].texte)
print(len(morceaux), "extraits pour", documents[0].source, "\n")
print(morceaux[2])

`rag.decouper` applique la stratégie choisie à tout le corpus et garde, pour chaque extrait, sa source, son titre, sa section et sa confidentialité.
La stratégie « sections » découpe d'abord selon les titres Markdown ; l'option `entete` ajoute en tête de chaque extrait le titre du document et de la section, ce qui aide la recherche. L'atelier 2.B compare les stratégies.

In [ ]:
extraits = rag.decouper(documents, strategie="sections", taille=800, chevauchement=100, entete=True)
print(len(extraits), "extraits\n")
print(extraits[30].reference())
print(extraits[30].texte)

## 5. Embeddings

Un modèle d'embeddings transforme un texte en vecteur : des textes de sens proche donnent des vecteurs proches. Le modèle utilisé ici est multilingue, open source et tourne en local. Le premier lancement le télécharge (0,22 Go).

In [ ]:
encodeur = rag.Encodeur()
print(encodeur.nom, "->", encodeur.dimension, "dimensions")

phrases = [
    "Livrez-vous à l'étranger ?",
    "Fictiva livre en Suisse et au Liechtenstein uniquement.",
    "Les factures sont payables à 30 jours net.",
    "Do you deliver abroad?",
]
vecteurs = encodeur.encoder(phrases)
pd.DataFrame(vecteurs @ vecteurs.T, index=phrases, columns=[p[:20] + "…" for p in phrases]).round(2)

Les vecteurs sont normalisés : leur produit scalaire est la **similarité cosinus**. La question et la phrase sur les zones de livraison sont proches, même sans mot commun ; la question anglaise aussi. La phrase sur les factures est loin.

## 6. Indexer

Chaque extrait est encodé et rangé dans une collection Qdrant, avec ses métadonnées (*payload*). Qdrant tourne ici en mémoire ; en production, c'est un serveur. La classe `rag.Index` construit aussi un index lexical BM25, utilisé à l'atelier 2.B.

In [ ]:
index = rag.Index(extraits, encodeur)
print(index.qdrant.count("fictiva").count, "vecteurs dans la collection")

## 7. Rechercher

In [ ]:
question = "Comment signaler un colis arrivé endommagé ?"
for e in index.dense(question, k=4):
    print(f"{e.score:.3f}  {e.reference()}")
    print("       ", e.texte[:160].replace("\n", " "), "…\n")

## 8. Générer une réponse citée

Le prompt système impose trois règles : répondre **uniquement** à partir des extraits, **citer** leur numéro, et dire qu'on ne sait pas quand les extraits ne suffisent pas. Les extraits sont placés entre balises et traités comme des données.

In [ ]:
print(rag.PROMPT_SYSTEME)

In [ ]:
trouves = index.dense(question, k=4)
prompt = rag.construire_prompt(question, trouves)
print(prompt[:1200], "…\n")

r = llm.demander(prompt, systeme=rag.PROMPT_SYSTEME, modele=MODELE)
print("Réponse :", r.texte)

✏️ Essayez la question « Quel est le délai d'intervention du contrat de maintenance Argent ? » : les extraits retrouvés contiennent-ils la ligne « Argent » du tableau ? Que répond le modèle ? Les ateliers 2.B et 2.C montrent comment améliorer la recherche.

## 9. Hors du corpus

Une bonne réponse, c'est aussi savoir dire qu'on ne sait pas.

In [ ]:
question = "Proposez-vous la location de véhicules utilitaires ?"
r = llm.demander(rag.construire_prompt(question, index.dense(question, k=4)), systeme=rag.PROMPT_SYSTEME, modele=MODELE)
print(r.texte)

## 10. Confidentialité : filtrer avant de chercher

La procédure interne contient le plafond des remises négociables avec les clients stratégiques. Comparons la recherche sur tout le corpus et la recherche limitée aux documents publics (filtre Qdrant sur le *payload*).

In [ ]:
question = "Quelle remise exceptionnelle pouvez-vous accorder à un client stratégique ?"
for public_seulement in (False, True):
    trouves = index.dense(question, k=4, public_seulement=public_seulement)
    r = llm.demander(rag.construire_prompt(question, trouves), systeme=rag.PROMPT_SYSTEME, modele=MODELE)
    print(f"Documents publics seulement : {public_seulement} | sources : {sorted({e.source for e in trouves})}")
    print(r.texte, "\n")

**À retenir.** Une consigne du type « ne divulgue pas les informations internes » ne protège rien : le modèle peut l'ignorer, et une injection de prompt peut la contourner. Le contrôle d'accès se fait **avant** la génération, en filtrant les documents selon les droits de l'utilisateur.

## 11. L'assistant documentaire

`rag.Assistant` assemble ces étapes : recherche limitée aux documents publics, génération citée, et liste des sources utilisées. C'est le livrable du jour, enrichi du reranking à l'atelier 2.C et exposé en API à l'atelier 2.D.

In [ ]:
assistant = rag.Assistant(index, modele=MODELE, mode="dense")
for q in ["Livrez-vous en France ?", "Qui paie les frais d'envoi d'un retour ?", "Comment configurer le RAID du serveur NAS NS-4 ?"]:
    r = assistant.repondre(q)
    print(f"Q : {q}\nR : {r.texte}\nSources : {r.sources()} ({r.latence_s:.1f} s)\n")

✏️ Posez vos propres questions. Cherchez-en une à laquelle l'assistant répond mal : est-ce un problème de **recherche** (les bons extraits ne sont pas retrouvés) ou de **génération** (les extraits sont là, mais la réponse est fausse) ? Affichez `r.extraits` pour le savoir.

## À retenir

- Le RAG ancre les réponses dans les documents de l'entreprise, sans réentraîner le modèle.
- La qualité dépend d'abord de la **lecture** des documents (PDF, tableaux) et du **découpage**.
- Les citations rendent chaque réponse vérifiable ; « je ne trouve pas » est une bonne réponse.
- La confidentialité se règle par filtrage à la recherche, jamais par le prompt.